# Import and Shared functions

In [ ]:
import os, sys
import math
import re
import pandas as pd
import numpy as np
import ephem
from scipy import signal
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from dateutil import parser as dateutil_parser
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', 'driver')))
import importlib
import kinematics
import control
importlib.reload(kinematics)
importlib.reload(control)
from kinematics import calc_parallactic_angle, azaltroll_to_theta, apply_mechanical_corrections, azaltroll_to_q, MountModelParams
from control import theta_to_jacobian, PecMode
from quaternion import Q as Quaternion

def r2_score(y, yhat):
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

def r2_quality(r2):
    return (
        "excellent" if r2>0.95 else 
        "good" if r2>0.8 else 
        "moderate" if r2>0.5 else 
        "weak" if r2>0.25 else 
        "poor")

def pec_quality_db(snr_db):
    return (
        "excellent" if snr_db > 30 else
        "good"      if snr_db > 20 else
        "moderate"  if snr_db > 10 else
        "weak"      if snr_db > 5  else
        "poor"
    )

def parse_val(v):
    v = v.strip()
    try:
        return float(v)
    except ValueError:
        return v
    
def parse_pecconfig(line):
    """Extract the PECCONFIG line emitted once per session, if present."""
    m = re.search(
        r'PECCONFIG mode,(\w+),n_harmonics,(\d+),T,([\d.]+),tau_sec,([\d.]+),min_dt_sec,([\d.]+)',
        line
    )
    if not m:
        return None
    mode, H, T, tau, min_dt = m.groups()
    return dict(mode=mode, H=int(H), T=float(T), tau=float(tau), min_dt=float(min_dt))

def parse_pec_line(body):
    """Parse one PECLOG body (everything after 'PECLOG') into a flat dict."""
    rec = {}
    for seg in body.split('|'):
        parts = [p.strip() for p in seg.split(',') if p.strip() != '']
        if not parts:
            continue
        label, vals = parts[0], parts[1:]

        if label == 'n' and len(vals) >= 3:
            rec['n']           = parse_val(vals[0])
            rec['ra_inhibit']  = vals[1]
            rec['dec_inhibit'] = vals[2]
        elif label == 'R2' and len(vals) >= 2:
            rec['ra_r2'],  rec['dec_r2']  = parse_val(vals[0]), parse_val(vals[1])
        elif label == 'rmse' and len(vals) >= 2:
            rec['ra_rmse'], rec['dec_rmse'] = parse_val(vals[0]), parse_val(vals[1])
        elif label == 'Rate' and len(vals) >= 2:
            rec['ra_pecrate'], rec['dec_pecrate'] = parse_val(vals[0]), parse_val(vals[1])
        elif label == 'Guide' and len(vals) >= 2:
            rec['ra_guide'], rec['dec_guide'] = parse_val(vals[0]), parse_val(vals[1])
        elif label == 'Accum' and len(vals) >= 2:
            rec['ra_accum'], rec['dec_accum'] = parse_val(vals[0]), parse_val(vals[1])
        elif label == 'Pos' and len(vals) >= 3:
            rec['az'], rec['alt'], rec['roll'] = parse_val(vals[0]), parse_val(vals[1]), parse_val(vals[2])
        elif label == 'P' and len(vals) >= 2:      # older scalar-P log format, if ever replayed
            rec['ra_P'], rec['dec_P'] = parse_val(vals[0]), parse_val(vals[1])
        elif label == 'RA_model':
            for i, v in enumerate(vals):
                rec[f'ra_a{i}'] = parse_val(v)          # ra_a0=dc, ra_a1=H1, ra_a2=H2, ...
        elif label == 'Dec_model':
            for i, v in enumerate(vals):
                rec[f'dec_a{i}'] = parse_val(v)
        elif label == 'lambda' and len(vals) >= 2:
            rec['ra_lambda'], rec['dec_lambda'] = parse_val(vals[0]), parse_val(vals[1])
    return rec

# Load data

In [ ]:
# Choose correct log path (last set log_path is what is used)
log_path = '../logs/alpaca.sga_omega_cent.log' # Omega Centuri 1st session
log_path = '../logs/alpaca.sga_lagoon.log'     # Lagoon Nebula 2nd session
log_path = '../logs/alpaca.sga_pulsepec.log'   # Pulse Guiding with PEC on Beehive cluster
log_path = '../logs/alpaca.sga_eagle.log'      # Sync Guiding with PEC on Eagle Nebula
log_path = '../logs/alpaca.sga_catspaw.log'      # Sync Guiding with PEC on Eagle Nebula
log_path = '../logs/alpaca.pecSlide.acrux2.log'
log_path = '../logs/vlogs/alpaca.log.4'       
log_path = '../logs/alpaca.h1.h0.log'
log_path = '../logs/alpaca.pec_rls_h2.log'
log_path = '../logs/alpaca.pec_ema.log'
#log_path = '../logs/alpaca.pecSin.chicken.log' # PHD2 guiding, somewhat ok

In [ ]:
if not os.path.exists(log_path):
    raise FileNotFoundError(f"log_path does not exist: {log_path!r}")
if os.path.getsize(log_path) == 0:
    raise ValueError(f"log_path exists but is empty (0 bytes): {log_path!r}")

n_lines = 0
rows = []
pec_config = None
with open(log_path) as f:
    for line in f:
        if 'PECCONFIG' in line:
            cfg = parse_pecconfig(line)
            if cfg is not None:
                pec_config = cfg
            continue
        if 'PECLOG' not in line:
            continue
        n_lines += 1
        ts   = line.split(' INFO ')[0].strip()
        body = line.split('PECLOG')[1].strip()
        rec  = parse_pec_line(body)
        rec['timestamp'] = ts
        rows.append(rec)

if n_lines == 0:
    raise ValueError(f"log_path opened but contained no PECLOG lines at all: {log_path!r}")

df = pd.DataFrame(rows)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['t_sec'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.total_seconds()
df = df.sort_values('t_sec').reset_index(drop=True)

# Fallback if this log predates PECCONFIG (older sessions) — infer what we can
if pec_config is None:
    ra_model_cols = [c for c in df.columns if re.fullmatch(r'ra_a\d+', c)]
    n_model_terms = len(ra_model_cols)
    if n_model_terms == 0:
        print("WARNING: no PECCONFIG line and no RA_model/Dec_model columns found — "
              "cannot determine mode reliably. Defaulting to RLS/H=0, verify manually.")
        pec_config = dict(mode='rls', H=0, T=34*60, tau=21*60, min_dt=0.05)
    else:
        pec_config = dict(mode='rls', H=max(n_model_terms - 1, 0), T=34*60, tau=21*60, min_dt=0.05)
    print("No PECCONFIG line found — inferred:", pec_config)
else:
    print("PEC config from log:", pec_config)

print(f"Duration: {df['t_sec'].iloc[-1]/3600:.2f} hours")
print(f"N samples: {len(df)}")
print(f"Az range: {df['az'].min():.3f} to {df['az'].max():.3f}")
print()
df.columns


In [ ]:
rows

# Optional Save to CSV

In [ ]:
df.to_csv('data.csv', index=False)

# Mount Orientation (Az, Alt, Roll) vs Time

In [ ]:
xdata = df['t_sec'] / 60
ydata = [
    #row, dataset,            name,                color
    (1,   df['az'],    'Azimuth (°)',  'royalblue'),
    (2,   df['alt'],   'Altitude (°)', 'orange'),
    (3,   df['roll'],  'Roll (°)',     'mediumseagreen'),
]
rows = len(set([row for row, y,name,color in ydata]))
fig = make_subplots(rows=rows, cols=1, shared_xaxes=True,
    subplot_titles=[name for row, y,name,color in ydata],
    vertical_spacing=0.12)

for (row, y, name, color) in ydata:
    fig.add_trace(go.Scatter(x=xdata, y=y,
        mode='lines', line=dict(color=color, width=0.8),
        name=name), row=row, col=1)
    fig.update_yaxes(title_text=name, row=row, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=3, col=1)
fig.update_layout(height=800, width=1200, template="plotly_dark",
        title='Range of Mount Orientation over session')
fig.show()

# Analysis of PECLOG

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from control import PecAxis, PecInhibit, PecMode

# ── calc INTERVAL (diagnostic only — no longer drives lambda/alpha directly) ──
ra_times = df.dropna(subset=['ra_guide'])['t_sec']
INTERVAL = ra_times.diff().dropna().median()

# ── instantiate from parsed session config ────────────────────────────────
T        = pec_config['T']
H        = pec_config['H']
TAU      = pec_config['tau']
MIN_DT   = pec_config['min_dt']
MODE     = PecMode(pec_config['mode']) if pec_config['mode'] in ('rls', 'ema') else PecMode.RLS
VAR      = 0.05
SSE      = 0.15
COLORS   = ['green', 'purple', 'orange', 'red', 'cyan', 'magenta']
DEG_S_TO_ARCMIN_HR = 3600 * 60   # deg/sec to arcmin/hr

ra  = PecAxis(T=T, n_harmonics=H, mode=MODE, tau=TAU, min_dt=MIN_DT)
dec = PecAxis(T=T, n_harmonics=H, mode=MODE, tau=TAU, min_dt=MIN_DT)
var_alpha = VAR
sse_alpha = SSE

ra.reset_seed(df.iloc[0].ra_accum / 60)
dec.reset_seed(df.iloc[0].dec_accum / 60)

show_harmonics = (MODE == PecMode.RLS and H > 0)

# ── replay ─────────────────────────────────────────────────────────────────────
records  = []

for i, row in df.iterrows():
    t         = row.t_sec
    ra_cumul  = row.ra_accum  / 60
    dec_cumul = row.dec_accum / 60

    # ingest_accum no longer takes lam — each axis derives lambda/alpha from its own dt
    ra.ingest_accum(ra_cumul, t, var_alpha, sse_alpha)
    dec.ingest_accum(dec_cumul, t, var_alpha, sse_alpha)

    records.append(dict(
        t_min    = t / 60,
        # predicted_rate(t) matches what the fixed eval_correction() applies in real
        # time — theta alone would show the stale value frozen at the last ingest.
        ra_rate  = ra.predicted_rate(t)  * DEG_S_TO_ARCMIN_HR,
        dec_rate = dec.predicted_rate(t) * DEG_S_TO_ARCMIN_HR,
        ra_drift = ra.dc_rate()   * DEG_S_TO_ARCMIN_HR,
        dec_drift= dec.dc_rate()  * DEG_S_TO_ARCMIN_HR,
        **{f'ra_h{h}':  ra.harmonic_rate(h)  * DEG_S_TO_ARCMIN_HR for h in range(1, H+1)},
        **{f'dec_h{h}': dec.harmonic_rate(h) * DEG_S_TO_ARCMIN_HR for h in range(1, H+1)},
        ra_r2    = ra.r2,
        dec_r2   = dec.r2,
        ra_rmse  = ra.rmse_arcmin(),
        dec_rmse = dec.rmse_arcmin(),
        ra_pred  = ra.predicted_accum(t)  * 60,
        dec_pred = dec.predicted_accum(t) * 60,
        ra_cumul = ra_cumul  * 60,
        dec_cumul= dec_cumul * 60,
        ra_resid_fit  = (ra_cumul*60)  - ra.predicted_accum(t)*60,
        dec_resid_fit = (dec_cumul*60) - dec.predicted_accum(t)*60,
        ra_lambda  = ra.lam,
        dec_lambda = dec.lam,
    ))

res = pd.DataFrame(records)
t   = res.t_min

# ── plot ───────────────────────────────────────────────────────────────────────
n_rows = 5 if show_harmonics else 4
row_titles = [
    'RA accum error and fit (arcmin)',            'Dec acumm error and fit (arcmin)',
    'RA PEC Rate and DC component (arcmin/hr)',   'Dec PEC Rate and DC component (arcmin/hr)',
]
if show_harmonics:
    row_titles += ['RA harmonic component (arcmin/hr)', 'Dec harmonic component (arcmin/hr)']
row_titles += ['RA fit R² quality', 'Dec fit R² quality',
               'RA fit residuals and rmse (arcmin)', 'Dec fit residuals and rmse (arcmin)']

fig = make_subplots(
    rows=n_rows, cols=2,
    shared_xaxes=True,
    subplot_titles=tuple(row_titles),
    vertical_spacing=0.06,
)

def add(row, col, traces):
    for tr in traces:
        fig.add_trace(tr, row=row, col=col)

# row 1 — cumul fit
add(1, 1, [
    go.Scatter(x=t, y=res.ra_cumul, name='RA actual',    opacity=0.6, line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.ra_pred,  name='RA predicted', line=dict(color='red', width=2)),
])
add(1, 2, [
    go.Scatter(x=t, y=res.dec_cumul, name='Dec actual',    opacity=0.6, line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.dec_pred,  name='Dec predicted', line=dict(color='red', width=2)),
])

# row 2 — rate vs drift (DC/mean rate; for EMA, ra_drift==ra_rate since there's no split)
add(2, 1, [
    go.Scatter(x=t, y=res.ra_rate,  name='RA PEC Rate', line=dict(color='orange')),
    go.Scatter(x=t, y=res.ra_drift, name='RA DC Component',       line=dict(color='#636EFA', dash='dash')),
])
add(2, 2, [
    go.Scatter(x=t, y=res.dec_rate,  name='Dec PEC Rate', line=dict(color='orange')),
    go.Scatter(x=t, y=res.dec_drift, name='Dec DC Component',       line=dict(color='#636EFA', dash='dash')),
])

next_row = 3
if show_harmonics:
    add(next_row, 1, [
        go.Scatter(x=t, y=res[f'ra_h{h}'], name=f'RA H{h} Component', line=dict(color=COLORS[(h-1) % len(COLORS)], dash='dash'))
        for h in range(1, H+1)
    ])
    add(next_row, 2, [
        go.Scatter(x=t, y=res[f'dec_h{h}'], name=f'Dec H{h} Component', line=dict(color=COLORS[(h-1) % len(COLORS)], dash='dash'))
        for h in range(1, H+1)
    ])
    next_row += 1

r2_row = next_row
add(r2_row, 1, [go.Scatter(x=t, y=res.ra_r2,  name='RA R²',  line=dict(color='royalblue'))])
add(r2_row, 2, [go.Scatter(x=t, y=res.dec_r2, name='Dec R²', line=dict(color='royalblue'))])
next_row += 1

resid_row = next_row
add(resid_row, 1, [
    go.Scatter(x=t, y=res.ra_resid_fit, name='RA Residuals', line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.ra_rmse,      name='RA rmse',      line=dict(color='red', dash='dash')),
])
add(resid_row, 2, [
    go.Scatter(x=t, y=res.dec_resid_fit, name='Dec Residuals', line=dict(color='royalblue')),
    go.Scatter(x=t, y=res.dec_rmse,      name='Dec rmse',      line=dict(color='red', dash='dash')),
])

fig.add_hrect(y0=-3, y1=0.5, fillcolor='rgba(255,0,0,0.15)', line_width=0, row=r2_row, col=1)
fig.add_hrect(y0=-3, y1=0.5, fillcolor='rgba(255,0,0,0.15)', line_width=0, row=r2_row, col=2)
fig.update_yaxes(range=[0.25, 1.0], row=r2_row, col=1)
fig.update_yaxes(range=[0.25, 1.0], row=r2_row, col=2)

mode_label = MODE.value.upper()
fig.update_layout(
    height=220 * n_rows,
    title=dict(
        text=(f'PEC Model (mode={mode_label}   Hₙ={H}   Period₁={T/60:.0f} min   '
              f'dt_med={INTERVAL:.1f}s   τ={TAU:.0f}s   sse_α={SSE}   var_α={VAR})'),
        x=0.5,
    ),
    hovermode='x unified', template="plotly_dark",
    legend=dict(groupclick='toggleitem'),
)
fig.update_xaxes(title_text='Time (minutes)', row=n_rows)
fig.show()

# Cumulative Pulse Guide Corrections vs Time

In [ ]:
xdata = df['t_sec'] / 60
ydata = [
    #row, dataset,            name,                color
    (1,   df['ra_accum'],  'Acumulated RA corrections (arcmin)',  'royalblue'),
    (2,   df['dec_accum'], 'Acumulated Dec corrections (arcmin)', 'orange'),
]
rows = len(set([row for row, y,name,color in ydata]))
fig = make_subplots(rows=rows, cols=1, shared_xaxes=True,
    subplot_titles=[name for row, y,name,color in ydata],
    vertical_spacing=0.12)


for (row, y, name, color) in ydata:
    fig.add_trace(go.Scatter(x=xdata, y=y,
        mode='lines', line=dict(color=color, width=0.8), hovertext=df['timestamp'],
        name=name), row=row, col=1)
    fig.update_yaxes(title_text=name, row=row, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=3, col=1)
fig.update_layout(height=800, width=1200, template="plotly_dark",
        title='Cumulative Auto Guide Corrections over session')
fig.show()

# PEC Rate and status vs Time

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

xdata = df['t_sec'] / 60

ydata = [
    # row, dataset,                title,                      color,       inhibit_col
    (1, df['ra_pecrate']*60,  'RA PEC Rate (arcmin/hr)',  'cyan', 'ra_inhibit'),
    (2, df['dec_pecrate']*60, 'DEC PEC Rate (arcmin/hr)', 'orange',    'dec_inhibit'),
    (1, df['ra_accum'],  'RA Accum (arcmin)',  'royalblue', None),
    (2, df['dec_accum'], 'DEC Accum (arcmin)', 'red', None)
]

rows = len(set(item[0] for item in ydata))

fig = make_subplots(
    rows=rows, cols=1, shared_xaxes=True, vertical_spacing=0.12,
    subplot_titles=[name for _, _, name, _, _ in ydata],
)

# colors for inhibit states
region_colors = {
    'VALID': None,
    'TOO_FEW_OBS': 'rgba(255,0,0,0.12)',
    'LOW_R2': 'rgba(255,255,0,0.15)',
    'HIGH_RMSE': 'rgba(0,255,0,0.15)',
    'NOT_CONVERGED': 'rgba(0,255,255,0.15)',
}

for (row, y, name, color, inhibit_col) in ydata:
    # main trace
    fig.add_trace(
        go.Scatter(
            x=xdata, y=y, mode='lines', name=name,
            line=dict(color=color, width=0.8), hovertext=df['timestamp'],
        ),
        row=row, col=1
    )

    # contiguous inhibit regions
    if inhibit_col:
        inhibit = df[inhibit_col].fillna('VALID')
        start_idx = 0
        current = inhibit.iloc[0]
        for i in range(1, len(inhibit)):
            if inhibit.iloc[i] != current:
                fill = region_colors.get(current)
                if fill:
                    fig.add_vrect(
                        x0=xdata.iloc[start_idx], x1=xdata.iloc[i],
                        fillcolor=fill, line_width=0, layer='below',
                        row=row, col=1,
                    )
                start_idx = i
                current = inhibit.iloc[i]

        # final region
        fill = region_colors.get(current)
        if fill:
            fig.add_vrect(
                x0=xdata.iloc[start_idx], x1=xdata.iloc[-1],
                fillcolor=fill, line_width=0, layer='below',
                row=row, col=1,
            )

    fig.update_yaxes(title_text=name, row=row, col=1)

fig.update_xaxes(title_text='Time (minutes)', row=rows, col=1)

fig.update_layout(
    height=800, width=1200, template="plotly_dark",
    title='Guide Correction and PEC Rates over session'
)

fig.show()

# Right Ascension Drift

In [ ]:
t = df['t_sec'].values
dec = df['ra_accum'].values
poly_coeffs, residuals, rank, sv, rcond = np.polyfit(t, dec, 1, full=True)
slope, intercept = poly_coeffs
dec_fit = np.polyval(poly_coeffs, t)
n = len(t)
rss = residuals[0] if len(residuals) > 0 else np.sum((dec - dec_fit)**2)
tss = np.sum((y - np.mean(dec))**2)
r2 = 1 - rss / tss if tss != 0 else float('nan')
rmse = np.sqrt(rss / n)
print(f"=== Right Ascension - Drift Fit Summary (N={n}) ===")
print(f"Slope (drift rate): {slope*3600:+8.3f} arcmin/hr")
print(f"Intercept:          {intercept:+8.3f} arcmin")
print(f"RMSE (noise):       {rmse:8.3f} arcmin")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")

# Declination Drift

In [ ]:
t = df['t_sec'].values
dec = df['dec_accum'].values
poly_coeffs, residuals, rank, sv, rcond = np.polyfit(t, dec, 1, full=True)
slope, intercept = poly_coeffs
dec_fit = np.polyval(poly_coeffs, t)
n = len(t)
rss = residuals[0] if len(residuals) > 0 else np.sum((dec - dec_fit)**2)
tss = np.sum((y - np.mean(dec))**2)
r2 = 1 - rss / tss if tss != 0 else float('nan')
rmse = np.sqrt(rss / n)
print(f"=== Declination - Drift Fit Summary (N={n}) ===")
print(f"Slope (drift rate): {slope*3600:+8.3f} arcmin/hr")
print(f"Intercept:          {intercept:+8.3f} arcmin")
print(f"RMSE (noise):       {rmse:8.3f} arcmin")
print(f"R² (fit quality):   {r2:8.3f} {r2_quality(r2)} fit")



# Right Ascension Period

In [ ]:
from scipy.signal import periodogram
import numpy as np

# RA: the periodic PEC signal
ra = df['ra_accum'].values

# Remove linear drift (same idea as DEC fit)
ra_detrended = ra - np.polyval(np.polyfit(t, ra, 1), t)

# Sampling frequency
fs = 1 / np.median(np.diff(t))

# Periodogram
freqs, power = periodogram(ra_detrended, fs=fs)

# Ignore zero frequency and periods > 100 min 
max_period_sec = 120 * 60
min_freq = 1 / max_period_sec
valid = (freqs >= min_freq)
freqs = freqs[valid]
power = power[valid]

# Peak detection
peak_idx = np.argmax(power)
peak_freq = freqs[peak_idx]
peak_power = power[peak_idx]

worm_period_sec = 1 / peak_freq

# Noise floor estimate (median is robust)
mask = np.ones_like(power, dtype=bool)
window = 3  # exclude ±3 bins around peak
mask[max(0, peak_idx-window):peak_idx+window+1] = False
noise_floor = np.median(power[mask])

# Signal-to-noise ratio
snr = peak_power / noise_floor if noise_floor > 0 else np.inf
snr_db = 10 * np.log10(peak_power / noise_floor)

# Estimate amplitude (rough, from detrended signal)
amp = (np.max(ra_detrended) - np.min(ra_detrended)) / 2

# Output
n = len(ra)

print(f"=== Right Ascension - PEC Analysis (N={n}) ===")
print(f"Worm period:        {worm_period_sec/60:8.3f} min")
print(f"Amplitude:          {amp*60:8.3f} arcmin")
print(f"Peak power:         {10*np.log10(peak_power):8.3f} dB (relative)")
print(f"Noise floor:        {10*np.log10(noise_floor):8.3f} dB (relative)")
print(f"SNR (periodicity):  {snr_db:8.2f} dB {pec_quality_db(snr_db)} signal")

In [ ]:
periods_min = 1 / freqs / 60
log_power = np.log10(power)
max_y = np.max(log_power)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=periods_min,
    y=log_power,
    mode='lines',
    name='log10(Power)'
))

fig.add_vline(x=worm_period_sec/60, line_dash="dash", line_color="red")

fig.update_layout(
    title="RA Periodogram (Log Power)",
    xaxis_title="Period (minutes)",
    yaxis_title="log10(Power)", 
    height=600, width=1200, template="plotly_dark",
)
max_y
fig.update_yaxes(range=[ -4, max_y])

fig.show()

# Declination Period

In [ ]:
from scipy.signal import periodogram
import numpy as np

# Dec: the periodic PEC signal
dec = df['dec_accum'].values

# Remove linear drift (same idea as DEC fit)
dec_detrended = dec - np.polyval(np.polyfit(t, dec, 1), t)

# Sampling frequency
fs = 1 / np.median(np.diff(t))

# Periodogram
freqs, power = periodogram(dec_detrended, fs=fs)

# Ignore zero frequency and periods > 100 min 
max_period_sec = 120 * 60
min_freq = 1 / max_period_sec
valid = (freqs >= min_freq)
freqs = freqs[valid]
power = power[valid]

# Peak detection
peak_idx = np.argmax(power)
peak_freq = freqs[peak_idx]
peak_power = power[peak_idx]

worm_period_sec = 1 / peak_freq

# Noise floor estimate (median is robust)
mask = np.ones_like(power, dtype=bool)
window = 3  # exclude ±3 bins around peak
mask[max(0, peak_idx-window):peak_idx+window+1] = False
noise_floor = np.median(power[mask])

# Signal-to-noise ratio
snr = peak_power / noise_floor if noise_floor > 0 else np.inf
snr_db = 10 * np.log10(peak_power / noise_floor)

# Estimate amplitude (rough, from detrended signal)
amp = (np.max(ra_detrended) - np.min(ra_detrended)) / 2

# Output
n = len(dec)

print(f"=== Declination - PEC Analysis (N={n}) ===")
print(f"Worm period:        {worm_period_sec/60:8.3f} min")
print(f"Amplitude:          {amp*60:8.3f} arcmin")
print(f"Peak power:         {10*np.log10(peak_power):8.3f} dB (relative)")
print(f"Noise floor:        {10*np.log10(noise_floor):8.3f} dB (relative)")
print(f"SNR (periodicity):  {snr_db:8.2f} dB {pec_quality_db(snr_db)} signal")

In [ ]:
periods_min = 1 / freqs / 60
log_power = np.log10(power)
max_y = np.max(log_power)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=periods_min,
    y=log_power,
    mode='lines',
    name='log10(Power)'
))

fig.add_vline(x=worm_period_sec/60, line_dash="dash", line_color="red")

fig.update_layout(
    title="Dec Periodogram (Log Power)",
    xaxis_title="Period (minutes)",
    yaxis_title="log10(Power)", 
    height=600, width=1200, template="plotly_dark",
)
max_y
fig.update_yaxes(range=[ -4, max_y])

fig.show()

# Notes